In [ ]:
import math 
def read_gfc(filename):
    C = {}
    S = {}
    Lmax = 0
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('GRCOF2'):
                parts = line.split()
                l = int(parts[1])
                m = int(parts[2])
                C[(l,m)] = float(parts[3])
                S[(l,m)] = float(parts[4])
                Lmax = max(Lmax, l)
    return C, S, Lmax

C,S,Lmax = read_gfc("GSM-2_2003001-2003031_GRAC_UTCSR_BA01_0600")
g_rho1 = []
for i in range(61):
    a = []
    for j in range(i+1):
        a.append(1000*C[(i,j)])
    b = []
    for j in range(i):
        b.append(1000*S[(i,j+1)])
    g_rho1.append(a[:]+b[:])
g_rho1[0][0] = 0.

In [ ]:
s = 0.51
List_Ln = [ ((k+2)*(k+1)/2+(k+1)*k/2)**(2*s) for k in range(61)]
norm = math.sqrt(sum([ coe_weight*sum([ a**2 for a in A]) for coe_weight,A in zip(List_Ln,g_rho1)]))
print(norm)

In [ ]:
import numpy as np
import math
import random
class g_rho():
    def __init__(self,number,s,r,g_rho1):
        self.number = number
        self.r = r
        self.s = s
        self.g_rho = g_rho1[:]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        
    def Centeral_C_K_update(self,x,k):
        if k == 0:
            self.C_K_List = [1]
        else:
            self.C_K_List = []
            self.C_K_2_List.append(0)
            j = k 
            lambda_j = k-j+1/2
            for k_1,k_2 in zip(self.C_K_1_List,self.C_K_2_List):
                self.C_K_List.append((2*(j+lambda_j-1)*x*k_1)/j - (j+2*lambda_j-2)*k_2/j)
                j = j - 1
                lambda_j = k-j+1/2
            self.C_K_List.append(1)
            self.C_K_2_List = self.C_K_1_List[:]
            self.C_K_1_List = self.C_K_List[:]
            
    def grho_cal(self,an_theta, an_phi):
        self.Computation_Yn_grho(an_theta,an_phi)
        return sum([sum([ a*b for a,b in zip(A,B)]) for A,B in zip(self.g_rho,self.Y_n_g_rho)])

    def Computation_Yn_grho(self,an_theta,an_phi):
        factorial_k = 1
        self.Y_n_g_rho = [[1]]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        for k in range(1,len(self.g_rho)):
            factorial_k *= k
            factorial_k_j_plus = [factorial_k]
            factorial_k_j_subtrac = [factorial_k]
            db_factorial = [1]
            sin_theta_k = [1]
            for j in range(1,k+1):
                sin_theta_k.append(sin_theta_k[-1]*math.sin(an_theta))
                factorial_k_j_plus.append(factorial_k_j_plus[-1]*(k+j))
                factorial_k_j_subtrac.append(factorial_k_j_subtrac[-1]/(k-j+1))
                db_factorial.append((2*j-1)*db_factorial[-1])
            self.Centeral_C_K_update(math.cos(an_theta),k)
            Y_List = [math.sqrt(2*(2*k+1)*factorial_k_j_subtrac[j]/factorial_k_j_plus[j])*db_factorial[j]*sin_theta_k[j]*self.C_K_List[j]  for j in range(k+1)]
            self.Y_n_g_rho.append([Y_List[j]*math.cos(j*an_phi) for j in range(k+1)]+[Y_List[j]*math.sin(j*an_phi) for j in range(1,k+1)])
    
    def construct_data(self):
        X = np.random.normal(size=(3))
        x,y,z = X/math.sqrt(sum(X**2))
        an_theta, an_phi = self.cartesian_to_spherical(x, y, z)
        Y = self.grho_cal(an_theta, an_phi)
        Z = Y*1000 + random.gauss(0,0.2)
        return [[an_theta, an_phi],Z,Y]

    def cartesian_to_spherical(self,x, y, z):
        theta = np.arccos(np.clip(z, -1.0, 1.0))
        phi = np.arctan2(y, x)  # returns in (-pi, pi]
        if  phi < 0:
            phi += 2*np.pi
        return theta, phi

DATA = []
s = 1.
r = 1.
G_rho = g_rho(10,s,r,g_rho1)
N = [int(10**(i/5)) for i in range(1,35)]
'''for i in range(6309630):
    DATA.append(G_rho.construct_data())
    if i in N:
        print(i)
'''

In [ ]:
#coding=gbk
import math
import random
import matplotlib.pyplot as plt
import time
import numpy as np
from decimal import *
getcontext().prec=36

class T_kernel_SGD():
    def __init__(self,s,Q,sigma,loss,r,number,g_rho1):
        self.s = s
        self.r = r
        self.number = number
        self.Ln = 0
        self.fn = [[0]]
        self.fn_average = [[0]]
        self.theta = theta
        self.gamma0 = 1
        self.t = 0
        self.epoch = 1
        self.Q = Q
        self.loss = loss
        self.sigma = sigma
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        self.g_rho = g_rho1
        

    def Centeral_C_K_update(self,x,k):
        if k == 0:
            self.C_K_List = [1]
        else:
            self.C_K_List = []
            self.C_K_2_List.append(0)
            j = k 
            lambda_j = k-j+1/2
            for k_1,k_2 in zip(self.C_K_1_List,self.C_K_2_List):
                self.C_K_List.append((2*(j+lambda_j-1)*x*k_1)/j - (j+2*lambda_j-2)*k_2/j)
                j = j - 1
                lambda_j = k-j+1/2
            self.C_K_List.append(1)
            self.C_K_2_List = self.C_K_1_List[:]
            self.C_K_1_List = self.C_K_List[:]
        

    def Computation_Yn(self,an_theta,an_phi):
        factorial_k = 1
        self.Y_n = [[1]]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        for k in range(1,self.Ln+1):
            factorial_k *= k
            factorial_k_j_plus = [factorial_k]
            factorial_k_j_subtrac = [factorial_k]
            db_factorial = [1]
            sin_theta_k = [1]
            for j in range(1,k+1):
                sin_theta_k.append(sin_theta_k[-1]*math.sin(an_theta))
                factorial_k_j_plus.append(factorial_k_j_plus[-1]*(k+j))
                factorial_k_j_subtrac.append(factorial_k_j_subtrac[-1]/(k-j+1))
                db_factorial.append((2*j-1)*db_factorial[-1])
            self.Centeral_C_K_update(math.cos(an_theta),k)
            Y_List = [math.sqrt(2*(2*k+1)*factorial_k_j_subtrac[j]/factorial_k_j_plus[j])*db_factorial[j]*sin_theta_k[j]*self.C_K_List[j]  for j in range(k+1)]
            self.Y_n.append([Y_List[j]*math.cos(j*an_phi) for j in range(k+1)]+[Y_List[j]*math.sin(j*an_phi) for j in range(1,k+1)])
            
        
    def Computation_fn(self,X):
        (an_theta,an_phi) = X
        self.Computation_Yn(an_theta,an_phi)
        S = sum([ sum([ a*b for a,b in zip(A,B)]) for A,B in zip(self.Y_n,self.fn)])
        return S

    def Computation_fn_average(self,X):
        (an_theta,an_phi) = X
        self.Computation_Yn(an_theta,an_phi)
        S = sum([ sum([ a*b for a,b in zip(A,B)]) for A,B in zip(self.Y_n,self.fn_average)])
        return S
    
    def derivative_loss(self,X,Y):
        if self.loss == 'least square':
            derivative = (self.Computation_fn(X)-Y)
        elif self.loss == 'logistic':
            term = math.exp(Y*self.Computation_fn(X))
            derivative = - Y / (1+term)
        elif self.loss == 'Cauchy':
            term = self.Computation_fn(X)-Y
            derivative = term/(self.sigma**2+term**2/2)
        elif self.loss == 'Huber':
            term = self.Computation_fn(X)-Y
            derivative = term/math.sqrt(term**2+1)
        elif self.loss == 'Welsch':
            term = self.Computation_fn(X)-Y
            derivative = math.exp(-term**2/(2*self.sigma**2))*(term/self.sigma**2)
        return derivative
    
    def iteration(self,X,Y):
        coe = -self.gamma0*((self.epoch)**(-self.t))*self.derivative_loss(X,Y)
        if (self.Ln+2)*(self.Ln+1)/2+(self.Ln+1)*self.Ln/2 < self.epoch**self.theta:
            self.Ln += 1
            self.fn.append([0 for i in range(2*self.Ln+1)])
            self.fn_average.append([0 for i in range(2*self.Ln+1)])
        self.Computation_Yn(X[0],X[1])
        List_Ln = [ ((k+2)*(k+1)/2+(k+1)*k/2)**(-2*self.s) for k in range(self.Ln+1)]
        self.fn = [ [ a+coe*coe_weight*b for a,b in zip(A,B) ] for A,B,coe_weight in zip(self.fn,self.Y_n,List_Ln)]
        self.fn_norm = math.sqrt(sum([ (coe_weight_1**(-1))*sum([ a**2 for a in A]) for coe_weight_1,A in zip(List_Ln,self.fn)]))
        if self.fn_norm > self.Q:
            zoom = self.Q/self.fn_norm
            self.fn = [[a*zoom for a in A] for A in self.fn]
        self.fn_average = [[(self.epoch*a+b)/(self.epoch+1) for a,b in zip(A,B)] for A,B in zip(self.fn_average,self.fn)]
        

    def grho_cal(self,an_theta, an_phi):
        self.Computation_Yn_grho(an_theta,an_phi)
        return sum([sum([ a*b for a,b in zip(A,B)]) for A,B in zip(self.g_rho,self.Y_n_g_rho)])

    def Computation_Yn_grho(self,an_theta,an_phi):
        factorial_k = 1
        self.Y_n_g_rho = [[1]]
        self.C_K_2_List = []
        self.C_K_1_List = [1]
        for k in range(1,len(self.g_rho)):
            factorial_k *= k
            factorial_k_j_plus = [factorial_k]
            factorial_k_j_subtrac = [factorial_k]
            db_factorial = [1]
            sin_theta_k = [1]
            for j in range(1,k+1):
                sin_theta_k.append(sin_theta_k[-1]*math.sin(an_theta))
                factorial_k_j_plus.append(factorial_k_j_plus[-1]*(k+j))
                factorial_k_j_subtrac.append(factorial_k_j_subtrac[-1]/(k-j+1))
                db_factorial.append((2*j-1)*db_factorial[-1])
            self.Centeral_C_K_update(math.cos(an_theta),k)
            Y_List = [math.sqrt(2*(2*k+1)*factorial_k_j_subtrac[j]/factorial_k_j_plus[j])*db_factorial[j]*sin_theta_k[j]*self.C_K_List[j]  for j in range(k+1)]
            self.Y_n_g_rho.append([Y_List[j]*math.cos(j*an_phi) for j in range(k+1)]+[Y_List[j]*math.sin(j*an_phi) for j in range(1,k+1)])
 
    def construct_data(self):
        if self.Uni == 'error':
            X = np.random.normal(size=(3))
            x,y,z = X/math.sqrt(sum(X**2))
            an_theta, an_phi = self.cartesian_to_spherical(x, y, z)
            Y = self.grho_cal(an_theta, an_phi)
            Z = Y + random.gauss(0,0.2)
            data = [[an_theta, an_phi],Z,Y]
        elif self.Uni == 'time':
            data = DATA[self.epoch]
        return data

    def cartesian_to_spherical(self,x, y, z):
        theta = np.arccos(np.clip(z, -1.0, 1.0))
        phi = np.arctan2(y, x)  # returns in (-pi, pi]
        if  phi < 0:
            phi += 2*np.pi
        return theta, phi

    def test(self):
        S1 = 0
        S2 = 0
        Number = 1000
        #end = time.perf_counter()
        for j in range(Number):
            Z = self.construct_data()
            S1 += (self.Computation_fn(Z[0])-Z[2])**2
            S2 += (self.Computation_fn_average(Z[0])-Z[2])**2
        #print('iteration:',i,S1/1000,S2/1000,self.L_n,end-start)
        return [S1/Number,S2/Number]

    def updating(self,gamma0,t,theta,Sum_epoch,NN,Uni):
        self.gamma0 = gamma0
        self.t = t
        self.Uni = Uni
        self.theta = theta
        self.Error_fn_ave = 0
        self.Error_fn = 0
        self.Time = 0
        self.Sum_epoch = Sum_epoch
        self.average_coe = 0
        start = time.perf_counter()
        for i in range(1,Sum_epoch+1):
            self.epoch = i
            Z = self.construct_data()
            self.iteration(Z[0],Z[-2])
        if self.Uni == 'time':
            end = time.perf_counter()
            self.Time = end-start
        elif self.Uni == 'error':
            S1,S2= self.test()
            L = len(self.fn_average)
            self.Error_fn_ave = sum([ sum([(x-y)**2 for x,y in zip(a,b)]) for a,b in zip(self.fn_average,G_rho.g_rho)]+[sum([x**2 for x in a]) for a in G_rho.g_rho[L:]])
            self.Error_fn = sum([ sum([(x-y)**2 for x,y in zip(a,b)]) for a,b in zip(self.fn,G_rho.g_rho)]+[sum([x**2 for x in a]) for a in G_rho.g_rho[L:]])
            Delta_list = [ (k*(k+1)) for k in range(60)]
            self.Error_fn_ave_delta = sum([ coe*sum([(x-y)**2 for x,y in zip(a,b)]) for a,b,coe in zip(self.fn_average,G_rho.g_rho,Delta_list)]+[coe*sum([x**2 for x in a]) for a,coe in zip(G_rho.g_rho[L:],Delta_list[L:])])
        List_Ln = [ ((k+2)*(k+1)/2+(k+1)*k/2)**(2*self.s) for k in range(self.Ln+1)]
        norm = math.sqrt(sum([ coe_weight*sum([ a**2 for a in A]) for coe_weight,A in zip(List_Ln,self.fn_average)]))
        print(i,self.Ln,self.Error_fn_ave,self.Error_fn,norm,self.fn_norm)
        print(self.Error_fn_ave_delta)
        #print(self.Ln,len(self.fn_average))
        #print(i,end-start)
        #self.Error.append(S2)
        #print(self.f_average_n)
        #print('K_n',self.K_n)

In [ ]:
N = [int(10**(i/5)) for i in range(1,26)]
s = 0.51
r = 1/2
gamma0 =1.5
number = 1
t = 2*r/(2*r+1)
theta = 0.28
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 20
T = 15
Loss = ['Cauchy']
for loss in Loss:
    err = []
    err_suffix = []
    err_ave = []
    err_suffix_ave = []
    err_suffix_delta = []
    err_suffix_ave_delta = []
    for i in range(T):
        E = []
        E_ave = []
        E_ave_delta = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,loss,r,number,g_rho1)
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E.append(t_kernel_SGD.Error_fn)
            E_ave.append(t_kernel_SGD.Error_fn_ave)
            E_ave_delta.append(t_kernel_SGD.Error_fn_ave_delta)
        err.append(E[:])
        err_suffix.append(E_ave[:])
        err_suffix_delta.append(E_ave_delta[:])
    for i in range(len(err[-1])):
        S = 0
        S1 = 0
        S2 = 0
        for j in range(T):
            S += err[j][i]
            S1 += err_suffix[j][i]
            S2 += err_suffix_delta[j][i]
        err_ave.append(S/T)
        err_suffix_ave.append(S1/T)
        err_suffix_ave_delta.append(S2/T)
    if loss == 'least square':
        value = 'least'
    else:
        value = loss
    print('err_t_kernel_SGD_'+value+'_s_1 =',err_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave =',err_suffix_ave)
    print('err_t_kernel1_SGD_ave_'+value+'_s_1_ave_delta =',err_suffix_ave_delta)
    print(' ')

In [ ]:
'''

new


'''

In [ ]:
import math 
def read_gfc(filename):
    C = {}
    S = {}
    Lmax = 0
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('GRCOF2'):
                parts = line.split()
                l = int(parts[1])
                m = int(parts[2])
                C[(l,m)] = float(parts[3])
                S[(l,m)] = float(parts[4])
                Lmax = max(Lmax, l)
    return C, S, Lmax

C,S,Lmax = read_gfc("GSM-2_2003091-2003120_GRAC_UTCSR_BA01_0600")
g_rho1 = []
for i in range(61):
    a = []
    for j in range(i+1):
        a.append(1000*C[(i,j)])
    b = []
    for j in range(i):
        b.append(1000*S[(i,j+1)])
    g_rho1.append(a[:]+b[:])
g_rho1[0][0] = 0.
DATA = []
s = 1.
r = 1.
G_rho = g_rho(10,s,r,g_rho1)
N = [int(10**(i/5)) for i in range(1,35)]

In [ ]:
N = [int(10**(i/5)) for i in range(1,26)]
s = 0.51
r = 1/2
gamma0 =1.5
number = 1
t = 2*r/(2*r+1)
theta = 0.28
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 20
T = 15
Loss = ['Cauchy']
for loss in Loss:
    err = []
    err_suffix = []
    err_ave = []
    err_suffix_ave = []
    err_suffix_delta = []
    err_suffix_ave_delta = []
    for i in range(T):
        E = []
        E_ave = []
        E_ave_delta = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,loss,r,number,g_rho1)
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E.append(t_kernel_SGD.Error_fn)
            E_ave.append(t_kernel_SGD.Error_fn_ave)
            E_ave_delta.append(t_kernel_SGD.Error_fn_ave_delta)
        err.append(E[:])
        err_suffix.append(E_ave[:])
        err_suffix_delta.append(E_ave_delta[:])
    for i in range(len(err[-1])):
        S = 0
        S1 = 0
        S2 = 0
        for j in range(T):
            S += err[j][i]
            S1 += err_suffix[j][i]
            S2 += err_suffix_delta[j][i]
        err_ave.append(S/T)
        err_suffix_ave.append(S1/T)
        err_suffix_ave_delta.append(S2/T)
    if loss == 'least square':
        value = 'least'
    else:
        value = loss
    print('err_t_kernel_SGD_'+value+'_s_1 =',err_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave =',err_suffix_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave_delta =',err_suffix_ave_delta)
    print(' ')

In [ ]:
'''
new new

'''

In [ ]:
import math 
def read_gfc(filename):
    C = {}
    S = {}
    Lmax = 0
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('GRCOF2'):
                parts = line.split()
                l = int(parts[1])
                m = int(parts[2])
                C[(l,m)] = float(parts[3])
                S[(l,m)] = float(parts[4])
                Lmax = max(Lmax, l)
    return C, S, Lmax

C,S,Lmax = read_gfc("GSM-2_2003182-2003212_GRAC_UTCSR_BA01_0600")
g_rho1 = []
for i in range(61):
    a = []
    for j in range(i+1):
        a.append(1000*C[(i,j)])
    b = []
    for j in range(i):
        b.append(1000*S[(i,j+1)])
    g_rho1.append(a[:]+b[:])
g_rho1[0][0] = 0.
DATA = []
s = 1.
r = 1.
G_rho = g_rho(10,s,r,g_rho1)
N = [int(10**(i/5)) for i in range(1,35)]

In [ ]:
for r in [ i/10+1 for i in range(41)]:
    List_Ln = [ ((k+2)*(k+1)/2+(k+1)*k/2)**(4*0.51*r) for k in range(len(g_rho1))]
    norm = math.sqrt(sum([ (coe_weight_1)*sum([ a**2 for a in A]) for coe_weight_1,A in zip(List_Ln,g_rho1)]))
    print(r,math.log(norm))

In [ ]:
N = [int(10**(i/5)) for i in range(1,26)]
s = 0.51
r = 1/2
gamma0 =1.5
number = 1
t = 2*r/(2*r+1)
theta = 0.28
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 20
T = 15
Loss = ['Cauchy']
for loss in Loss:
    err = []
    err_suffix = []
    err_ave = []
    err_suffix_ave = []
    err_suffix_delta = []
    err_suffix_ave_delta = []
    for i in range(T):
        E = []
        E_ave = []
        E_ave_delta = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,loss,r,number,g_rho1)
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E.append(t_kernel_SGD.Error_fn)
            E_ave.append(t_kernel_SGD.Error_fn_ave)
            E_ave_delta.append(t_kernel_SGD.Error_fn_ave_delta)
        err.append(E[:])
        err_suffix.append(E_ave[:])
        err_suffix_delta.append(E_ave_delta[:])
    for i in range(len(err[-1])):
        S = 0
        S1 = 0
        S2 = 0
        for j in range(T):
            S += err[j][i]
            S1 += err_suffix[j][i]
            S2 += err_suffix_delta[j][i]
        err_ave.append(S/T)
        err_suffix_ave.append(S1/T)
        err_suffix_ave_delta.append(S2/T)
    if loss == 'least square':
        value = 'least'
    else:
        value = loss
    print('err_t_kernel_SGD_'+value+'_s_1 =',err_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave =',err_suffix_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave_delta =',err_suffix_ave_delta)
    print(' ')

In [ ]:
'''
new new new 

'''

In [ ]:
import math 
def read_gfc(filename):
    C = {}
    S = {}
    Lmax = 0
    
    with open(filename, 'r') as f:
        for line in f:
            if line.startswith('GRCOF2'):
                parts = line.split()
                l = int(parts[1])
                m = int(parts[2])
                C[(l,m)] = float(parts[3])
                S[(l,m)] = float(parts[4])
                Lmax = max(Lmax, l)
    return C, S, Lmax

C,S,Lmax = read_gfc("GSM-2_2003274-2003304_GRAC_UTCSR_BA01_0600")
g_rho1 = []
for i in range(61):
    a = []
    for j in range(i+1):
        a.append(1000*C[(i,j)])
    b = []
    for j in range(i):
        b.append(1000*S[(i,j+1)])
    g_rho1.append(a[:]+b[:])
g_rho1[0][0] = 0.
DATA = []
s = 1.
r = 1.
G_rho = g_rho(10,s,r,g_rho1)
N = [int(10**(i/5)) for i in range(1,35)]

In [ ]:
for r in [ i/10+1 for i in range(41)]:
    List_Ln = [ ((k+2)*(k+1)/2+(k+1)*k/2)**(4*0.51*r) for k in range(len(g_rho1))]
    norm = math.sqrt(sum([ (coe_weight_1)*sum([ a**2 for a in A]) for coe_weight_1,A in zip(List_Ln,g_rho1)]))
    print(r,math.log(norm))

In [ ]:
N = [int(10**(i/5)) for i in range(1,26)]
s = 0.51
r = 1/2
gamma0 =1.5
number = 1
t = 2*r/(2*r+1)
theta = 0.28
Sum_epoch = int(N[-1]+1)
sigma = 1.
Q = 20
T = 15
Loss = ['Cauchy']
for loss in Loss:
    err = []
    err_suffix = []
    err_ave = []
    err_suffix_ave = []
    err_suffix_delta = []
    err_suffix_ave_delta = []
    for i in range(T):
        E = []
        E_ave = []
        E_ave_delta = []
        for k in N:
            t_kernel_SGD = T_kernel_SGD(s+10**(-8)*i+10**(-12)*np.log(k),Q,sigma,loss,r,number,g_rho1)
            t_kernel_SGD.updating(gamma0,t,theta,k,N,'error')
            E.append(t_kernel_SGD.Error_fn)
            E_ave.append(t_kernel_SGD.Error_fn_ave)
            E_ave_delta.append(t_kernel_SGD.Error_fn_ave_delta)
        err.append(E[:])
        err_suffix.append(E_ave[:])
        err_suffix_delta.append(E_ave_delta[:])
    for i in range(len(err[-1])):
        S = 0
        S1 = 0
        S2 = 0
        for j in range(T):
            S += err[j][i]
            S1 += err_suffix[j][i]
            S2 += err_suffix_delta[j][i]
        err_ave.append(S/T)
        err_suffix_ave.append(S1/T)
        err_suffix_ave_delta.append(S2/T)
    if loss == 'least square':
        value = 'least'
    else:
        value = loss
    print('err_t_kernel_SGD_'+value+'_s_1 =',err_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave =',err_suffix_ave)
    print('err_t_kernel_SGD_ave_'+value+'_s_1_ave_delta =',err_suffix_ave_delta)
    print(' ')